# Compare bioinformatics tools

This notebook takes the cleaned datsets (long and phyloseq) and explores the differences between bioinformatics tools. 
- Comparisons between:
    - 3 denoising methods (DADA2, VSEARCH and MetaBEAT)
    - 5 taxonomy assignment methods (RDP, BLAST LCA, MAPseq, MetaBEAT, VSEARCH LCA) 
- Comparisons are repeated across three datasets which have known fish species from historical visual surveying (Ring trial, Marchamley, and Windermere)
- Compares ASVs, sequences, and taxa (species, genus and family)
- Explores concordance with visual data 

## Set-up

In [0]:
# Install packages
installed <- rownames(installed.packages())

ensure_cran <- function(pkgs, repos = "https://www.stats.bris.ac.uk/R/") {
  to_install <- setdiff(pkgs, installed)
  if (length(to_install)) {
    install.packages(to_install, dependencies = TRUE, repos = repos)
  }
  invisible(lapply(pkgs, function(p)
    suppressPackageStartupMessages(library(p, character.only = TRUE))
  ))
}

ensure_bioc <- function(pkgs) {
  if (!requireNamespace("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
  }
  to_install <- setdiff(pkgs, installed)
  if (length(to_install)) {
    BiocManager::install(to_install, ask = FALSE, update = FALSE)
  }
  invisible(lapply(pkgs, function(p)
    suppressPackageStartupMessages(library(p, character.only = TRUE))
  ))
}

## CRAN packages
cran_pkgs <- c("Rcpp", "devtools", "ggplot2", "optparse", "dplyr", "seqinr", "tidyverse", "ggtext")
ensure_cran(cran_pkgs)

## Bioconductor packages
bioc_pkgs <- c("Biostrings", "ShortRead", "dada2", "phyloseq", "microbiome")
ensure_bioc(bioc_pkgs)

In [0]:
%python
pip install biopython

In [0]:
#overall
master_long_worms_df <- read.csv("Data/Processed/master_long_worms_df_filtered.csv")
phylo_eDNA <- readRDS("Data/Processed/phylo_eDNA.RDS")

#fish
master_long_worms_df_fish <- read.csv("Data/Processed/master_long_worms_df_filtered_fish.csv")
phylo_eDNA_fish <- readRDS("Data/Processed/phylo_eDNA_fish.RDS")

#nonfish
master_long_worms_df_nonfish <- read.csv("Data/Processed/master_long_worms_df_filtered_nonfish.csv")
phylo_eDNA_nonfish <- readRDS("Data/Processed/phylo_eDNA_nonfish.RDS")

#visual data
visual_data <- read.csv(file = "Data/Processed/visual_data_tidy.csv")

In [0]:
# Universal labels defined
dataset_names <- c(
    "marchamley" = "Marchamley",
    "ringtrial_sean" = "Ring Trial",
    "windermere_2017" = "Windermere"
  )

## Total reads and ASVs/OTUs

Number of reads and unique ASV should be the same within the same denoising algoritm. 

In [0]:
denoise_summary <- master_long_worms_df %>%
  group_by(denoise_method, taxonomy_method) %>%
  summarise(

    total_reads_qc = sum(reads, na.rm = TRUE),
    total_unique_asvs = n_distinct(asv),

    pct_reads_family = round(
      100 * sum(
        reads[
          !is.na(family) &
          !tolower(family) %in% c("unclassified", "na", "unassigned", "unknown", "")
        ],
        na.rm = TRUE
      ) / sum(reads, na.rm = TRUE),
      1
    ),

    pct_reads_genus = round(
      100 * sum(
        reads[
          !is.na(genus) &
          !tolower(genus) %in% c("unclassified", "na", "unassigned", "unknown", "")
        ],
        na.rm = TRUE
      ) / sum(reads, na.rm = TRUE),
      1
    ),

    pct_reads_species = round(
      100 * sum(
        reads[
          !is.na(species) &
          !tolower(species) %in% c("unclassified", "na", "unassigned", "unknown", "")
        ],
        na.rm = TRUE
      ) / sum(reads, na.rm = TRUE),
      1
    ),

    asvs_family = n_distinct(
      asv[
        !is.na(family) &
        !tolower(family) %in% c("unclassified", "na", "unassigned", "unknown", "")
      ]
    ),

    asvs_genus = n_distinct(
      asv[
        !is.na(genus) &
        !tolower(genus) %in% c("unclassified", "na", "unassigned", "unknown", "")
      ]
    ),

    asvs_species = n_distinct(
      asv[
        !is.na(species) &
        !tolower(species) %in% c("unclassified", "na", "unassigned", "unknown", "")
      ]
    ),

    .groups = "drop"
  )

# save
write.csv(
  denoise_summary,
  "Results/01_All/reads_asvs_summary.csv",
  row.names = FALSE
)

In [0]:
denoise_summary # the 0.01% threshold how now been applied which is why they don't add up exactly anymore

## Unassigned ASVs/OTUs & taxa

#### All data

In [0]:
input_data <- master_long_worms_df
source("Scripts/11_check_unassigned.R")
write.csv(unassigned_summary_table, "Results/01_All/unassigned_summary_table.csv", row.names = FALSE)

In [0]:
# capitalise and group
asv_data <- subset(unassigned_summary_table, metric == "Reads") %>%
subset(taxonomy_method != "MetaBEAT") %>% # remove for now as can only compare this with other metabeat
  mutate(
    level = str_to_title(level)
  ) %>%
  group_by(level, taxonomy_method, dataset) %>%
  mutate(
    point_type = case_when(
      percent_unassigned == max(percent_unassigned) ~ "Highest",
      percent_unassigned == min(percent_unassigned) ~ "Lowest",
      TRUE ~ "Middle"
    )
  ) %>%
  ungroup()

# plot (red = highest unassigned)
unassigned_asv_denoise <- ggplot(asv_data,
       aes(x = denoise_method,
           y = percent_unassigned,
           group = dataset,
           linetype = dataset,
           shape = dataset)) +
  geom_line(colour = "grey40") +
geom_point(aes(colour = point_type), size = 3) +
scale_colour_manual(
  name = "Point type",
  values = c(
    "Highest" = "red",
    "Lowest" = "green2",
    "Middle" = "yellow2"
  )
) +
  scale_x_discrete(
    labels = c(
      "dada2" = "DADA2",
      "metabeat" = "MetaBEAT",
      "vsearch" = "VSEARCH"
    )
  ) +
  facet_grid(level ~ taxonomy_method) +
  theme_bw() +
  labs(
    x = "Denoising tool",
    y = "Unassigned ASVs/OTUs normalised by reads (%)",
    linetype = "Dataset",
    shape = "Dataset"
  ) +
  scale_linetype_discrete(labels = dataset_names) +
  scale_shape_discrete(labels = dataset_names)

save
  ggsave(
  "Results/01_All/unassigned_asv_denoise.png",
  unassigned_asv_denoise,
  width = 12,
  height = 10,
  dpi = 300
)

In [0]:
unassigned_asv_denoise

In [0]:
# capitalise and group
asv_data2 <- subset(unassigned_summary_table, metric == "Reads") %>%
subset(taxonomy_method != "MetaBEAT") %>% # remove for now as can only compare this with other metabeat
  mutate(
    level = str_to_title(level)
  ) %>%
  group_by(level, denoise_method, dataset) %>%
  mutate(
    point_type = case_when(
      percent_unassigned == max(percent_unassigned) ~ "Highest",
      percent_unassigned == min(percent_unassigned) ~ "Lowest",
      TRUE ~ "Middle"
    )
  ) %>%
  ungroup()

# plot
unassigned_asv_taxaAssign <- ggplot(asv_data2,
       aes(x = taxonomy_method,
           y = percent_unassigned,
           group = dataset,
           linetype = dataset,
           shape = dataset)) +
  geom_line(colour = "grey40") +
  geom_point(aes(colour = point_type), size = 3) +
scale_colour_manual(
  name = "Point type",
  values = c(
    "Highest" = "red",
    "Lowest" = "green2",
    "Middle" = "yellow2"
  )
) +
  facet_grid(
    level ~ denoise_method,
    labeller = labeller(
      denoise_method = c(
        dada2 = "DADA2",
        metabeat = "MetaBEAT",
        vsearch = "VSEARCH"
      )
    )
  ) +
  theme_bw() +
  labs(
    x = "Taxonomic assignment tool",
    y = "Unassigned ASVs/OTUs normalised by reads (%)",
    linetype = "Dataset",
    shape = "Dataset"
  ) +
  scale_linetype_discrete(labels = dataset_names) +
  scale_shape_discrete(labels = dataset_names)

#save
ggsave(
  "Results/01_All/unassigned_asv_taxaAssign.png",
  unassigned_asv_taxaAssign,
  width = 11,
  height = 10,
  dpi = 300
)

In [0]:
unassigned_asv_taxaAssign

In [0]:

# capitalise and group
taxa_data <- subset(unassigned_summary_table, metric == "taxa") %>%
subset(taxonomy_method != "MetaBEAT") %>% # remove for now as can only compare this with other metabeat
  mutate(
    level = str_to_title(level)
  ) %>%
  group_by(level, taxonomy_method, dataset) %>%
  mutate(
    point_type = case_when(
      percent_unassigned == max(percent_unassigned) ~ "Highest",
      percent_unassigned == min(percent_unassigned) ~ "Lowest",
      TRUE ~ "Middle"
    )
  ) %>%
  ungroup()

# plot (red = highest unassigned)
unassigned_taxa_denoise <- ggplot(taxa_data,
       aes(x = denoise_method,
           y = percent_unassigned,
           group = dataset,
           linetype = dataset,
           shape = dataset)) +
  geom_line(colour = "grey40") +
  geom_point(aes(colour = point_type), size = 3) +
scale_colour_manual(
  name = "Point type",
  values = c(
    "Highest" = "red",
    "Lowest" = "green2",
    "Middle" = "yellow2"
    )
) +
  scale_x_discrete(
    labels = c(
      "dada2" = "DADA2",
      "metabeat" = "MetaBEAT",
      "vsearch" = "VSEARCH"
    )
  ) +
  facet_grid(level ~ taxonomy_method) +
  theme_bw() +
  labs(
    x = "Denoising tool",
    y = "Percent unassigned taxa",
    linetype = "Dataset",
    shape = "Dataset"
  ) +
  scale_linetype_discrete(labels = dataset_names) +
  scale_shape_discrete(labels = dataset_names)

#save
ggsave(
  "Results/01_All/unassigned_taxa_denoise.png",
  unassigned_taxa_denoise,
  width = 12,
  height = 10,
  dpi = 300
)

In [0]:
unassigned_taxa_denoise

In [0]:
# capitalise and group
taxa_data2 <- subset(unassigned_summary_table, metric == "taxa") %>%
subset(taxonomy_method != "MetaBEAT") %>% # remove for now as can only compare this with other metabeat
  mutate(
    level = str_to_title(level)
  ) %>%
  group_by(level, denoise_method, dataset) %>%
  mutate(
    point_type = case_when(
      percent_unassigned == max(percent_unassigned) ~ "Highest",
      percent_unassigned == min(percent_unassigned) ~ "Lowest",
      TRUE ~ "Middle"
    )
  ) %>%
  ungroup()

# plot
unassigned_taxa_taxaAssign <- ggplot(taxa_data2,
       aes(x = taxonomy_method,
           y = percent_unassigned,
           group = dataset,
           linetype = dataset,
           shape = dataset)) +
  geom_line(colour = "grey40") +
  geom_point(aes(colour = point_type), size = 3) +
scale_colour_manual(
  name = "Point type",
  values = c(
    "Highest" = "red",
    "Lowest" = "green2",
    "Middle" = "yellow2"
    )
) +
  facet_grid(
    level ~ denoise_method,
    labeller = labeller(
      denoise_method = c(
        dada2 = "DADA2",
        metabeat = "MetaBEAT",
        vsearch = "VSEARCH"
      )
    )
  ) +
  theme_bw() +
  labs(
    x = "Taxonomic assignment tool",
    y = "Percent unassigned taxa",
    linetype = "Dataset",
    shape = "Dataset"
  ) +
  scale_linetype_discrete(labels = dataset_names) +
  scale_shape_discrete(labels = dataset_names)

#save
ggsave(
  "Results/01_All/unassigned_taxa_taxaAssign.png",
  unassigned_taxa_taxaAssign,
  width = 11,
  height = 10,
  dpi = 300
)

In [0]:
unassigned_taxa_taxaAssign

#### Fish only

In [0]:
%skip
input_data <- master_long_worms_df_fish
source("Scripts/11_check_unassigned.R")
write.csv(unassigned_summary_table, "Results/02_fish/unassigned_summary_table.csv", row.names = FALSE)

In [0]:
%skip
# capitalise and group
asv_data_fish <- subset(unassigned_summary_table, metric == "Reads") %>%
subset(taxonomy_method != "MetaBEAT") %>% # remove for now as can only compare this with other metabeat
  mutate(
    level = str_to_title(level)
  ) %>%
  group_by(level, taxonomy_method, dataset) %>%
  mutate(
    point_type = case_when(
      percent_unassigned == max(percent_unassigned) ~ "Highest",
      percent_unassigned == min(percent_unassigned) ~ "Lowest",
      TRUE ~ "Middle"
    )
  ) %>%
  ungroup()

# plot (red = highest unassigned)
unassigned_asv_denoise_fish <- ggplot(asv_data_fish,
       aes(x = denoise_method,
           y = percent_unassigned,
           group = dataset,
           linetype = dataset,
           shape = dataset)) +
  geom_line(colour = "grey40") +
geom_point(aes(colour = point_type), size = 3) +
scale_colour_manual(
  name = "Point type",
  values = c(
    "Highest" = "red",
    "Lowest" = "green2",
    "Middle" = "yellow2"
    )
) +
  scale_x_discrete(
    labels = c(
      "dada2" = "DADA2",
      "metabeat" = "MetaBEAT",
      "vsearch" = "VSEARCH"
    )
  ) +
  facet_grid(level ~ taxonomy_method) +
  theme_bw() +
  labs(
    x = "Denoising tool",
    y = "Percent unassigned ASVs / OTUs (%)",
    linetype = "Dataset",
    shape = "Dataset"
  ) +
  scale_linetype_discrete(labels = dataset_names) +
  scale_shape_discrete(labels = dataset_names)

#save
  ggsave(
  "Results/02_fish/unassigned_asv_denoise_fish.png",
  unassigned_asv_denoise_fish,
  width = 9,
  height = 10,
  dpi = 300
)

#print
unassigned_asv_denoise_fish

In [0]:
%skip
# capitalise and group
asv_data2_fish <- subset(unassigned_summary_table, metric == "ASV") %>%
  mutate(
    level = str_to_title(level)
  ) %>%
  group_by(level, denoise_method, dataset) %>%
  mutate(
    point_type = case_when(
      percent_unassigned == max(percent_unassigned) ~ "Highest",
      percent_unassigned == min(percent_unassigned) ~ "Lowest",
      TRUE ~ "Middle"
    )
  ) %>%
  ungroup()

# plot
unassigned_asv_taxaAssign_fish <- ggplot(asv_data2_fish,
       aes(x = taxonomy_method,
           y = percent_unassigned,
           group = dataset,
           linetype = dataset,
           shape = dataset)) +
  geom_line(colour = "grey40") +
geom_point(aes(colour = point_type), size = 3) +
scale_colour_manual(
  name = "Point type",
  values = c(
    "Highest" = "red",
    "Lowest" = "green2",
    "Middle" = "yellow2"
    )
) +
  facet_grid(
    level ~ denoise_method,
    labeller = labeller(
      denoise_method = c(
        dada2 = "DADA2",
        metabeat = "MetaBEAT",
        vsearch = "VSEARCH"
      )
    )
  ) +
  theme_bw() +
  labs(
    x = "Taxonomic assignment tool",
    y = "Percent unassigned ASVs / OTUs (%)",
    linetype = "Dataset",
    shape = "Dataset"
  ) +
  scale_linetype_discrete(labels = dataset_names) +
  scale_shape_discrete(labels = dataset_names)

#save
ggsave(
  "Results/02_fish/unassigned_asv_taxaAssign_fish.png",
  unassigned_asv_taxaAssign_fish,
  width = 9,
  height = 10,
  dpi = 300
)

#print
unassigned_asv_taxaAssign_fish

In [0]:
%skip
# capitalise and group
taxa_data_fish <- subset(unassigned_summary_table, metric == "taxa") %>%
subset(taxonomy_method != "MetaBEAT") %>% # remove for now as can only compare this with other metabeat
  mutate(
    level = str_to_title(level)
  ) %>%
  group_by(level, taxonomy_method, dataset) %>%
  mutate(
    point_type = case_when(
      percent_unassigned == max(percent_unassigned) ~ "Highest",
      percent_unassigned == min(percent_unassigned) ~ "Lowest",
      TRUE ~ "Other"
    )
  ) %>%
  ungroup()

# plot (red = highest unassigned)
unassigned_taxa_denoise_fish <- ggplot(taxa_data_fish,
       aes(x = denoise_method,
           y = percent_unassigned,
           group = dataset,
           linetype = dataset,
           shape = dataset)) +
  geom_line(colour = "grey40") +
  geom_point(aes(colour = point_type), size = 3) +
scale_colour_manual(
  name = "Point type",
  values = c(
    "Highest" = "red",
    "Lowest" = "green2",
    "Middle" = "yellow2"
    )
) +
  scale_x_discrete(
    labels = c(
      "dada2" = "DADA2",
      "metabeat" = "MetaBEAT",
      "vsearch" = "VSEARCH"
    )
  ) +
  facet_grid(level ~ taxonomy_method) +
  theme_bw() +
  labs(
    x = "Denoising tool",
    y = "Percent unassigned taxa",
    linetype = "Dataset",
    shape = "Dataset"
  ) +
  scale_linetype_discrete(labels = dataset_names) +
  scale_shape_discrete(labels = dataset_names)

#save
ggsave(
  "Results/02_fish/unassigned_taxa_denoise_fish.png",
  unassigned_taxa_denoise_fish,
  width = 9,
  height = 10,
  dpi = 300
)

#print
unassigned_taxa_denoise_fish

In [0]:
%skip
# capitalise and group
taxa_data2_fish <- subset(unassigned_summary_table, metric == "taxa") %>%
  mutate(
    level = str_to_title(level)
  ) %>%
  group_by(level, denoise_method, dataset) %>%
  mutate(
    point_type = case_when(
      percent_unassigned == max(percent_unassigned) ~ "Highest",
      percent_unassigned == min(percent_unassigned) ~ "Lowest",
      TRUE ~ "Other"
    )
  ) %>%
  ungroup()

# plot
unassigned_taxa_taxaAssign_fish <- ggplot(taxa_data2_fish,
       aes(x = taxonomy_method,
           y = percent_unassigned,
           group = dataset,
           linetype = dataset,
           shape = dataset)) +
  geom_line(colour = "grey40") +
  geom_point(aes(colour = point_type), size = 3) +
scale_colour_manual(
  name = "Point type",
  values = c(
    "Highest" = "red",
    "Lowest" = "green2",
    "Middle" = "yellow2"
    )
) +
  facet_grid(
    level ~ denoise_method,
    labeller = labeller(
      denoise_method = c(
        dada2 = "DADA2",
        metabeat = "MetaBEAT",
        vsearch = "VSEARCH"
      )
    )
  ) +
  theme_bw() +
  labs(
    x = "Taxonomic assignment tool",
    y = "Percent unassigned taxa",
    linetype = "Dataset",
    shape = "Dataset"
  ) +
  scale_linetype_discrete(labels = dataset_names) +
  scale_shape_discrete(labels = dataset_names)

#save
ggsave(
  "Results/02_fish/unassigned_taxa_taxaAssign_fish.png",
  unassigned_taxa_taxaAssign_fish,
  width = 9,
  height = 10,
  dpi = 300
)

#print
unassigned_taxa_taxaAssign_fish

### Explore taxonomy

For all taxa.

In [0]:
print(paste("Species:", length(get_taxa_unique(phylo_eDNA, "species"))))
print((paste("Genus:", length(get_taxa_unique(phylo_eDNA, "genus")))))
print((paste("Family:", length(get_taxa_unique(phylo_eDNA, "family")))))
print((paste("Class:", length(get_taxa_unique(phylo_eDNA, "class")))))
print((paste("Phylum:", length(get_taxa_unique(phylo_eDNA, "phylum")))))

Just for fish.

In [0]:
print(paste("Species:", length(get_taxa_unique(phylo_eDNA_fish, "species"))))
print((paste("Genus:", length(get_taxa_unique(phylo_eDNA_fish, "genus")))))
print((paste("Family:", length(get_taxa_unique(phylo_eDNA_fish, "family")))))
print((paste("Class:", length(get_taxa_unique(phylo_eDNA_fish, "class")))))
print((paste("Phylum:", length(get_taxa_unique(phylo_eDNA_fish, "phylum")))))

#### Summary bar plots

All by class.

In [0]:
summary_df <- master_long_worms_df %>%
  distinct(dataset, denoise_method, taxonomy_method, class, species, fun_group) %>%
  dplyr::count(dataset, denoise_method, taxonomy_method, class, fun_group)

In [0]:
taxa_summary_plots <- summary_df %>%
  subset(class != "NA") %>% # remove NA columns
  mutate(class = factor(class, levels = rev(sort(unique(class))))) %>%
  split(.$dataset) %>%
  map(~ ggplot(
    .x,
    aes(x = class, y = n, fill = fun_group)
  ) +
    geom_col() +
    geom_text(
      aes(label = n),
      hjust = -0.1,
      size = 3
    ) +
    coord_flip() +
    expand_limits(y = max(.x$n) * 1.1) +
facet_grid(
  denoise_method ~ taxonomy_method,
  scales = "free_y",
  labeller = labeller(
    denoise_method = c(
      dada2 = "DADA2",
      metabeat = "MetaBEAT",
      vsearch = "VSEARCH"
    )
  )
)+
    theme_bw() +
    labs(
      title = stringr::str_to_sentence(unique(.x$dataset)),
      y = "Number of species",
      x = "Class",
      fill = "Functional group"
    )
  )

# fix to save to correct folders
folder_lookup <- c(
  "ringtrial_sean" = "RingTrial_Sean"
)

# save
iwalk(
  taxa_summary_plots,
  ~ {
    folder <- if (.y %in% names(folder_lookup)) {
      folder_lookup[.y]
    } else {
      stringr::str_to_sentence(.y)
    }
    ggsave(
      filename = paste0("Results/", folder, "/", folder, "_class_counts.png"),
      plot = .x,
      width = 12,
      height = 12,
      dpi = 300
    )
  }
)

In [0]:
taxa_summary_plots[[1]]

In [0]:
taxa_summary_plots[[2]]

In [0]:
taxa_summary_plots[[3]]

Colour by denoising method.

In [0]:
taxa_summary_plots_denoise <- summary_df %>%
  mutate(
    class = factor(class, levels = rev(sort(unique(class)))),
    denoise_method = dplyr::recode(
      denoise_method,
      "dada2" = "DADA2",
      "metabeat" = "MetaBEAT",
      "vsearch" = "VSEARCH"
    )
  ) %>%
  subset( class != "NA") %>% # remove NA columns
  subset( class != "unknown") %>%
  split(.$dataset) %>%
  map(~ {
    
    dodge <- position_dodge(width = 0.9)
    
    ggplot(
      .x,
      aes(
        x = class,
        y = n,
        fill = denoise_method
      )
    ) +
      geom_col(position = dodge) +
      geom_text(
        aes(label = n),
        position = dodge,
        hjust = -0.1,
        size = 3
      ) +
      coord_flip() +
      expand_limits(y = max(.x$n) * 1.1) +
      facet_grid(
        fun_group ~ taxonomy_method,
        scales = "free_y"
      ) +
      theme_bw() +
      labs(
        title = stringr::str_to_sentence(unique(.x$dataset)),
        y = "Number of species",
        x = "Class",
        fill = "Denoising method"
      )
  })

# fix to save to correct folders
iwalk(
  taxa_summary_plots_denoise,
  ~ {
    folder <- if (.y %in% names(folder_lookup)) {
      folder_lookup[.y]
    } else {
      stringr::str_to_sentence(.y)
    }
    ggsave(
      filename = paste0("Results/", folder, "/", folder, "_class_counts_denoise.png"),
      plot = .x,
      width = 14,
      height = 13,
      dpi = 300
    )
  }
)

In [0]:
taxa_summary_plots_denoise[[1]]

In [0]:
taxa_summary_plots_denoise[[2]]

In [0]:
taxa_summary_plots_denoise[[3]]

Fish by species.

In [0]:
# DNA summary
summary_df_fish <- master_long_worms_df_fish %>%
  distinct(dataset, denoise_method, taxonomy_method, family, genus, species) %>%
  dplyr::count(dataset, denoise_method, taxonomy_method, genus, species)

#==============================================================
# MATCH LOOKUP
#==============================================================

# Exact species matches
exact_matches <- summary_df_fish %>%
  semi_join(
    visual_data %>%
      distinct(dataset, taxa_name_clean),
    by = c(
      "dataset",
      "species" = "taxa_name_clean"
    )
  ) %>%
  distinct(dataset, species) %>%
  mutate(match_type = "exact")

# Genus matches
genus_matches <- summary_df_fish %>%
  semi_join(
    visual_data %>%
      distinct(dataset, genus_worms),
    by = c(
      "dataset",
      "genus" = "genus_worms"
    )
  ) %>%
  distinct(dataset, species) %>%
  mutate(match_type = "genus")

# Prioritise exact matches
match_lookup <- bind_rows(
  exact_matches,
  genus_matches
) %>%
  arrange(desc(match_type == "exact")) %>%
  distinct(dataset, species, .keep_all = TRUE)

#==============================================================
# DNA DATA FOR PLOTTING
#==============================================================

summary_df_fish_plot <- summary_df_fish %>%
  left_join(
    match_lookup,
    by = c("dataset", "species")
  ) %>%
  mutate(
    match_type = replace_na(match_type, "none")
  )

#==============================================================
# VISUAL FACET DATA
#==============================================================

visual_plot_data <- visual_data %>%
  distinct(
    dataset,
    species = taxa_name_clean
  ) %>%
  filter(
    !is.na(species),
    species != "na"
  ) %>%
  mutate(
    denoise_method = "Visual",
    taxonomy_method = "Visual",
    genus = NA_character_,
    n = 1,
    match_type = "visual"
  )

# Combine DNA and visual records
summary_df_fish_plot <- bind_rows(
  summary_df_fish_plot,
  visual_plot_data
)

In [0]:
taxa_summary_plots_visual <- summary_df_fish_plot %>%
  filter(
    !is.na(species),
    species != "na"
  ) %>%
  mutate(
    denoise_method = recode(
      denoise_method,
      "dada2" = "DADA2",
      "metabeat" = "MetaBEAT",
      "vsearch" = "VSEARCH"
    )
  ) %>%
  split(.$dataset) %>%
  map(~{

    plot_data <- .x %>%
      mutate(

        # Format species names
        species_label = species %>%
          str_replace_all("_", " ") %>%
          str_to_lower(),

        species_label = paste0(
          "<i>",
          str_replace(
            species_label,
            "^([a-z]+)",
            ~str_to_sentence(.x)
          ),
          "</i>"
        ),

        taxonomy_method = factor(
          taxonomy_method,
          levels = c(
            "Visual",
            sort(
              unique(
                taxonomy_method[
                  taxonomy_method != "Visual"
                ]
              )
            )
          )
        ),

        species_label = factor(
          species_label,
          levels = rev(
            sort(
              unique(species_label)
            )
          )
        ),

        denoise_method = factor(
          denoise_method,
          levels = c(
            "Visual",
            "DADA2",
            "MetaBEAT",
            "VSEARCH"
          )
        )
      )

    ggplot(
      plot_data,
      aes(
        x = denoise_method,
        y = species_label
      )
    ) +

      #--------------------------------------------------------
      # Visual-supported DNA detections
      #--------------------------------------------------------
      geom_tile(
        data = filter(
          plot_data,
          taxonomy_method != "Visual",
          match_type %in% c("exact", "genus")
        ),
        aes(fill = match_type),
        alpha = 0.5,
        width = 0.9,
        height = 0.9
      ) +
      scale_fill_manual(
        values = c(
          exact = "lightgreen",
          genus = "grey70"
        ),
        name = "Visual match"
      )+

      #--------------------------------------------------------
      # DNA detections
      #--------------------------------------------------------
      geom_point(
        data = filter(
          plot_data,
          taxonomy_method != "Visual"
        ),
        aes(
          size = n,
          colour = denoise_method
        ),
        alpha = 0.8
      ) +

      #--------------------------------------------------------
      # Visual detections
      #--------------------------------------------------------
      geom_point(
        data = filter(
          plot_data,
          taxonomy_method == "Visual"
        ),
        colour = "darkred",
        shape = 4,
        stroke = 1.2,
        size = 3
      ) +

      facet_grid(
        ~taxonomy_method,
        scales = "free_x",
        space = "free_x"
      ) +

      scale_y_discrete(
        drop = FALSE
      ) +

      guides(
        size = "none"
      ) +

      theme_bw() +

      labs(
        title = str_to_sentence(
          unique(plot_data$dataset)
        ),
        x = "Detection method",
        y = "Species",
        colour = "Method"
      ) +

      theme(
        axis.text.x = element_text(
          angle = 45,
          hjust = 1
        ),

        axis.text.y = ggtext::element_markdown(
          size = 8
        ),

        strip.background = element_rect(
          fill = "grey90"
        )
      )

  })

#==============================================================
# SAVE
#==============================================================

purrr::iwalk(
  taxa_summary_plots_visual,
  ~ggsave(
    filename = paste0(
      "Results/taxa_summary_bubble_visual_",
      .y,
      ".png"
    ),
    plot = .x,
    width = 10,
    height = 9,
    dpi = 300
  )
)

In [0]:
taxa_summary_plots_visual[[1]]

In [0]:
taxa_summary_plots_visual[[2]]

In [0]:
taxa_summary_plots_visual[[3]]

In [0]:
visual_not_edna_species<- visual_data %>%
  distinct(dataset, taxa_name_clean) %>%
  anti_join(
    master_long_worms_df_fish %>%
      distinct(dataset, species),
    by = c(
      "dataset",
      "taxa_name_clean" = "species"
    )
  )

visual_not_edna_species

Non-fish by species.

In [0]:
summary_df_nonfish <- master_long_worms_df_nonfish %>%
  distinct(dataset, denoise_method, taxonomy_method, family, genus, species) %>%
  dplyr::count(dataset, denoise_method, taxonomy_method, genus, species)

In [0]:
taxa_summary_plots_nonfish <- summary_df_nonfish %>%
  filter(species != "na") %>%
  mutate(
    species = factor(
      species,
      levels = rev(sort(unique(species)))
    ),
    denoise_method = recode(
      denoise_method,
      "dada2" = "DADA2",
      "metabeat" = "MetaBEAT",
      "vsearch" = "VSEARCH"
    )
  ) %>%
  split(.$dataset) %>%
  purrr::map(~{

    ggplot(
      .x,
      aes(
        x = denoise_method,
        y = species,
        size = n,
        colour = denoise_method
      )
    ) +
      geom_point(alpha = 0.8, size = 2.5) +
      facet_wrap(
        ~ taxonomy_method,
        nrow = 1
      ) +
      guides(size = "none") +
      theme_bw() +
      labs(
        title = stringr::str_to_sentence(unique(.x$dataset)),
        x = "Denoising method",
        y = "Species",
        colour = "Denoising method"
      ) +
      theme(
        axis.text.x = element_text(angle = 45, hjust = 1),
        axis.text.y = element_text(size = 8),
        strip.background = element_rect(fill = "grey90")
      )

  })

  purrr::iwalk(
  taxa_summary_plots_nonfish,
  ~ ggsave(
    paste0(
      "Results/taxa_summary_bubble_nonfish",
      .y,
      ".png"
    ),
    .x,
    width = 10,
    height = 9,
    dpi = 300
  )
)

In [0]:
taxa_summary_plots_nonfish[[1]]

In [0]:
taxa_summary_plots_nonfish[[2]]

In [0]:
taxa_summary_plots_nonfish[[3]]

### Explore community composition 

Let's make some broad family stacked plots across all methods and datasets.

### All data

In [0]:
ps_march <- subset_samples(phylo_eDNA, dataset == "marchamley")
ps_march <- prune_taxa(taxa_sums(ps_march) > 0, ps_march)
ps_family <- tax_glom(ps_march, taxrank = "family")
ps_family_rel <- transform_sample_counts(ps_family, function(x) x / sum(x))

p_march <- plot_bar(ps_family_rel, x= "sampleid", fill = "family") +
  ggplot2::labs(title = "Family-level composition (Marchamley)") +
  ggplot2::theme_bw()+
  theme(legend.position = "none",
  axis.text.x = element_text(angle = 90, hjust = 1))+
  facet_grid(taxonomy_method ~ denoise_method)

print(p_march)

ggsave(
  filename = "Results/Marchamley/Marchamley_Family_barplot.png",
  plot = p_march,
  width = 27,      # wide for many samples
  height = 14,      
  dpi = 300        # high resolution
)

In [0]:
ps_wind <- subset_samples(phylo_eDNA, dataset == "windermere_2017")
ps_wind <- prune_taxa(taxa_sums(ps_wind) > 0, ps_wind)
ps_family <- tax_glom(ps_wind, taxrank = "family")
ps_family_rel <- transform_sample_counts(ps_family, function(x) x / sum(x))

p_wind <- plot_bar(ps_family_rel, x= "sampleid", fill = "family") +
  ggplot2::labs(title = "Family-level composition (Windermere)") +
  ggplot2::theme_bw()+
  theme(legend.position = "none",
  axis.text.x = element_text(angle = 90, hjust = 1))+
  facet_grid(taxonomy_method ~ denoise_method)

print(p_wind)

ggsave(
  filename = "Results/Windermere_2017/Winderemere_Family_barplot.png",
  plot = p_wind,
  width = 27,      # wide for many samples
  height = 14,      
  dpi = 300        # high resolution
)

In [0]:
ps_ring <- subset_samples(phylo_eDNA, dataset == "ringtrial_sean")
ps_ring <- prune_taxa(taxa_sums(ps_ring) > 0, ps_ring)
ps_family <- tax_glom(ps_ring, taxrank = "family")
ps_family_rel <- transform_sample_counts(ps_family, function(x) x / sum(x))

p_ring <- plot_bar(ps_family_rel, x= "sampleid", fill = "family") +
  ggplot2::labs(title = "Family-level composition (Ring Trial)") +
  ggplot2::theme_bw()+
  theme(legend.position = "none",
  axis.text.x = element_text(angle = 90, hjust = 1))+
  facet_grid(taxonomy_method ~ denoise_method)

print(p_ring)

ggsave(
  filename = "Results/RingTrial_Sean/RingTrial_Family_barplot.png",
  plot = p_ring,
  width = 27,      # wide for many samples
  height = 14,      
  dpi = 300        # high resolution
)

### Fish only

In [0]:
ps_march_fish <- subset_samples(phylo_eDNA_fish, dataset == "marchamley")
ps_march_fish <- prune_taxa(taxa_sums(ps_march_fish) > 0, ps_march_fish)
ps_family <- tax_glom(ps_march_fish, taxrank = "family")
ps_family_rel <- transform_sample_counts(ps_family, function(x) x / sum(x))

p_march_fish <- plot_bar(ps_family_rel, x= "sampleid", fill = "family") +
  ggplot2::labs(title = "Family-level composition (Marchamley)") +
  ggplot2::theme_bw()+
  theme(
  axis.text.x = element_text(angle = 90, hjust = 1))+
  facet_grid(taxonomy_method ~ denoise_method)

print(p_march_fish)

ggsave(
  filename = "Results/02_fish/Marchamley_Family_barplot_fish.png",
  plot = p_march_fish,
  width = 27,      # wide for many samples
  height = 14,      
  dpi = 300        # high resolution
)

In [0]:
ps_wind_fish <- subset_samples(phylo_eDNA_fish, dataset == "windermere_2017")
ps_wind_fish <- prune_taxa(taxa_sums(ps_wind_fish) > 0, ps_wind_fish)
ps_family <- tax_glom(ps_wind_fish, taxrank = "family")
ps_family_rel <- transform_sample_counts(ps_family, function(x) x / sum(x))

p_wind_fish <- plot_bar(ps_family_rel, x= "sampleid", fill = "family") +
  ggplot2::labs(title = "Family-level composition (Winderemere)") +
  ggplot2::theme_bw()+
  theme(
  axis.text.x = element_text(angle = 90, hjust = 1))+
  facet_grid(taxonomy_method ~ denoise_method)

print(p_wind_fish)

ggsave(
  filename = "Results/02_fish/Winderemere_Family_barplot.png",
  plot = p_wind_fish,
  width = 27,      # wide for many samples
  height = 14,      
  dpi = 300        # high resolution
)

In [0]:
ps_ring_fish <- subset_samples(phylo_eDNA, dataset == "ringtrial_sean")
ps_ring_fish <- prune_taxa(taxa_sums(ps_ring_fish) > 0, ps_ring_fish)
ps_family <- tax_glom(ps_ring_fish, taxrank = "family")
ps_family_rel <- transform_sample_counts(ps_family, function(x) x / sum(x))

p_ring_fish <- plot_bar(ps_family_rel, x= "sampleid", fill = "family") +
  ggplot2::labs(title = "Family-level composition (Ring Trial)") +
  ggplot2::theme_bw()+
  theme(  axis.text.x = element_text(angle = 90, hjust = 1))+
  facet_grid(taxonomy_method ~ denoise_method)

print(p_ring_fish)

ggsave(
  filename = "Results/02_fish/RingTrial_Family_barplot_fish.png",
  plot = p_ring_fish,
  width = 27,      # wide for many samples
  height = 14,      
  dpi = 300        # high resolution
)

### Explore diversity

In [0]:
alpha_div <- estimate_richness(
  phylo_eDNA,
  measures = c("Observed", "Shannon")
)

meta <- data.frame(sample_data(phylo_eDNA))
df <- cbind(alpha_div, meta)

p_richness <- ggplot(df, aes(x = denoise_method, y = Observed, fill = denoise_method)) +
  geom_boxplot() +
  facet_grid(dataset ~ taxonomy_method) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(x = "Denoising Method",
       y = "Richness Diversity")

p_richness

In [0]:
richness_all_taxamethod <- ggplot(
  df,
  aes(
    x = denoise_method,
    y = Observed,
    fill = taxonomy_method
  )
) +
  geom_boxplot(
    width = 0.75,
    outlier.alpha = 0.5
  ) +
  facet_wrap(~dataset, scales = "free_y",
  labeller = labeller(
    dataset = c(
      marchamley = "Marchamley",
      ringtrial_sean = "Ring trial",
      windermere_2017 = "Windermere"
))) +
  scale_fill_manual(
    values = c(
      "#0072B2", # blue
      "#D55E00", # vermillion
      "#009E73", # green
      "#CC79A7", # purple
      "#E69F00"  # orange
    ),
    name = "Taxonomic classifier"
  ) +
  scale_x_discrete(
    labels = c(
      dada2 = "DADA2",
      metabeat = "MetaBEAT",
      vsearch = "VSEARCH"
    )
  ) +
  labs(
    x = "Denoising tool",
    y = "Richness"
  ) +
  theme_bw() +
  theme(
    axis.text.x = element_text(size = 11, angle = 45, hjust = 1),
    legend.position = "right"
  )

ggsave(
  filename = "Results/01_All/richness_all_taxamethod.png",
  plot = richness_all_taxamethod,
  width = 15,      # wide for many samples
  height = 8,      
  dpi = 300        # high resolution
)

richness_all_taxamethod

In [0]:
ggplot(df, aes(x = taxonomy_method, y = Observed, fill = taxonomy_method)) +
  geom_boxplot() +
  theme_bw()

### Similarity

Overall - Jaccard

In [0]:
physeq_beta <- prune_samples(sample_sums(phylo_eDNA_fish) > 0, phylo_eDNA_fish) # Remove empty samples
physeq_beta_PA <- microbiome::transform(physeq_beta,"pa") # Presence/absence transform
ord.nmds.jaccard <- ordinate(physeq_beta_PA, method = "NMDS", distance = "jaccard", k = 3) # NMDS on Jaccard distance

In [0]:
# get stress value
stress_value_jaccard <- ord.nmds.jaccard$stress

#plot
NMDS_denoise_jaccard <- plot_ordination(physeq_beta_PA, ord.nmds.jaccard, color="denoise_method")+
  theme_classic()+ 
  annotate("text", x = Inf, y = Inf, label = paste("Stress =", round(stress_value_jaccard, 3)), 
           size = 4, hjust = 1.1, vjust = 1.5)+
  stat_ellipse(aes(group = denoise_method), type = "t", level = 0.95, linetype = 1, size = 0.5)+
  labs(color = "Denoise Method") +
  theme(legend.position = "bottom") +
  theme(legend.position = "none")+ 
  guides(colour = guide_legend(nrow = 2, byrow = TRUE))

NMDS_denoise_jaccard

In [0]:
NMDS_classifer_jaccard <- plot_ordination(physeq_beta_PA, ord.nmds.jaccard, color="taxonomy_method")+
  theme_classic()+ 
  annotate("text", x = Inf, y = Inf, label = paste("Stress =", round(stress_value_jaccard, 3)), 
           size = 4, hjust = 1.1, vjust = 1.5)+
  stat_ellipse(aes(group = taxonomy_method), type = "t", level = 0.95, linetype = 1, size = 0.5)+
  labs(color = "Taxonomic Classifer Method") +
  theme(legend.position = "bottom") +
  theme(legend.position = "none")+ 
  guides(colour = guide_legend(nrow = 2, byrow = TRUE)) 

NMDS_classifer_jaccard

Windermere - Jaccard

In [0]:
physeq_beta_wind <- subset_samples(phylo_eDNA_fish, dataset == "windermere_2017") # Windermere only
physeq_beta_wind <- prune_samples(sample_sums(physeq_beta_wind) > 0, physeq_beta_wind) # Remove empty samples
physeq_beta_wind_PA <- microbiome::transform(physeq_beta_wind,"pa") # Presence/absence transform
ord.nmds.jaccard_wind <- ordinate(physeq_beta_wind_PA, method = "NMDS", distance = "jaccard", k = 3) # NMDS on Jaccard distance

In [0]:
# get stress value
stress_value_jaccard_wind <- ord.nmds.jaccard_wind$stress

#plot
NMDS_denoise_jaccard_wind <- plot_ordination(physeq_beta_wind_PA, ord.nmds.jaccard_wind, color="denoise_method")+
  theme_classic()+ 
  annotate("text", x = Inf, y = Inf, label = paste("Stress =", round(stress_value_jaccard_wind, 3)), 
           size = 4, hjust = 1.1, vjust = 1.5)+
  stat_ellipse(aes(group = denoise_method), type = "t", level = 0.95, linetype = 1, size = 0.5)+
  labs(color = "Denoise Method") +
  theme(legend.position = "bottom") +
  theme(legend.position = "none")+ 
  guides(colour = guide_legend(nrow = 2, byrow = TRUE)) +
  xlim(-0.2,0.2)

NMDS_denoise_jaccard_wind

In [0]:
NMDS_classifer_jaccard_wind <- plot_ordination(physeq_beta_wind_PA, ord.nmds.jaccard_wind, color="taxonomy_method")+
  theme_classic()+ 
  annotate("text", x = Inf, y = Inf, label = paste("Stress =", round(stress_value_jaccard_wind, 3)), 
           size = 4, hjust = 1.1, vjust = 1.5)+
  stat_ellipse(aes(group = taxonomy_method), type = "t", level = 0.95, linetype = 1, size = 0.5)+
  labs(color = "Taxonomic Classifer Method") +
  theme(legend.position = "bottom") +
  #theme(legend.position = "none")+ 
  guides(colour = guide_legend(nrow = 2, byrow = TRUE)) +
  xlim(-0.2,0.2)

NMDS_classifer_jaccard_wind

Ring trial

In [0]:
physeq_beta_ring <- subset_samples(phylo_eDNA_fish, dataset == "ringtrial_sean") # ringermere only
physeq_beta_ring <- prune_samples(sample_sums(physeq_beta_ring) > 0, physeq_beta_ring) # Remove empty samples
physeq_beta_ring_PA <- microbiome::transform(physeq_beta_ring,"pa") # Presence/absence transform
ord.nmds.jaccard_ring <- ordinate(physeq_beta_ring_PA, method = "NMDS", distance = "jaccard",  k = 3) # NMDS on Jaccard distance

In [0]:
# get stress value
stress_value_jaccard_ring <- ord.nmds.jaccard_ring$stress

#plot
NMDS_denoise_jaccard_ring <- plot_ordination(physeq_beta_ring_PA, ord.nmds.jaccard_ring, color="denoise_method")+
  theme_classic()+ 
  annotate("text", x = Inf, y = Inf, label = paste("Stress =", round(stress_value_jaccard_ring, 3)), 
           size = 4, hjust = 1.1, vjust = 1.5)+
  stat_ellipse(aes(group = denoise_method), type = "t", level = 0.95, linetype = 1, size = 0.5)+
  labs(color = "Denoise Method") +
  theme(legend.position = "bottom") +
  theme(legend.position = "none")+ 
  guides(colour = guide_legend(nrow = 2, byrow = TRUE)) 

NMDS_denoise_jaccard_ring

In [0]:
NMDS_classifer_jaccard_ring <- plot_ordination(physeq_beta_ring_PA, ord.nmds.jaccard_ring, color="taxonomy_method")+
  theme_classic()+ 
  annotate("text", x = Inf, y = Inf, label = paste("Stress =", round(stress_value_jaccard_ring, 3)), 
           size = 4, hjust = 1.1, vjust = 1.5)+
  stat_ellipse(aes(group = taxonomy_method), type = "t", level = 0.95, linetype = 1, size = 0.5)+
  labs(color = "Taxonomic Classifer Method") +
  theme(legend.position = "bottom") +
  #theme(legend.position = "none")+ 
  guides(colour = guide_legend(nrow = 2, byrow = TRUE)) 

NMDS_classifer_jaccard_ring

Marchamley

In [0]:
physeq_beta_march <- subset_samples(phylo_eDNA_fish, dataset == "marchamley") # marchermere only
physeq_beta_march <- prune_samples(sample_sums(physeq_beta_march) > 0, physeq_beta_march) # Remove empty samples
physeq_beta_march_PA <- microbiome::transform(physeq_beta_march,"pa") # Presence/absence transform
ord.nmds.jaccard_march <- ordinate(physeq_beta_march_PA, method = "NMDS", distance = "jaccard") # NMDS on Jaccard distance

In [0]:
# get stress value
stress_value_jaccard_march <- ord.nmds.jaccard_march$stress

#plot
NMDS_denoise_jaccard_march <- plot_ordination(physeq_beta_march_PA, ord.nmds.jaccard_march, color="denoise_method")+
  theme_classic()+ 
  annotate("text", x = Inf, y = Inf, label = paste("Stress =", round(stress_value_jaccard_march, 3)), 
           size = 4, hjust = 1.1, vjust = 1.5)+
  stat_ellipse(aes(group = denoise_method), type = "t", level = 0.95, linetype = 1, size = 0.5)+
  labs(color = "Denoise Method") +
  theme(legend.position = "bottom") +
  theme(legend.position = "none")+ 
  guides(colour = guide_legend(nrow = 2, byrow = TRUE)) +
  xlim(-25,-15)

NMDS_denoise_jaccard_march

In [0]:
NMDS_classifer_jaccard_march <- plot_ordination(physeq_beta_march_PA, ord.nmds.jaccard_march, color="taxonomy_method")+
  theme_classic()+ 
  annotate("text", x = Inf, y = Inf, label = paste("Stress =", round(stress_value_jaccard_march, 3)), 
           size = 4, hjust = 1.1, vjust = 1.5)+
  stat_ellipse(aes(group = taxonomy_method), type = "t", level = 0.95, linetype = 1, size = 0.5)+
  labs(color = "Taxonomic Classifer Method") +
  theme(legend.position = "bottom") +
  #theme(legend.position = "none")+ 
  guides(colour = guide_legend(nrow = 2, byrow = TRUE)) +
  xlim(-25,-15)

NMDS_classifer_jaccard_march

#### Pairwise similarity

In [0]:
jaccard_dist <- phyloseq::distance(phylo_eDNA_fish, method = "jaccard")

# format matrix to long
jaccard_dist_mat <- as.matrix(jaccard_dist)
longData<-reshape2::melt(jaccard_dist_mat)
longData<-longData[longData$value!=0,]

In [0]:
metadf <- data.frame(sample_data(phylo_eDNA_fish))
metadf_method <- metadf %>% subset(select = c("fullID", "denoise_method", "taxonomy_method", "dataset"))

In [0]:
# Join longData with metadf_method to get Var1_denoise
longData <- longData %>%
  left_join(
    metadf_method %>% 
      dplyr::select(fullID, Var1_denoise = denoise_method),
    by = c("Var1" = "fullID")
  )

# Join longData with metadf_method again to get Var2_denoise
longData <- longData %>%
  left_join(
    metadf_method %>% 
      dplyr::select(fullID, Var2_denoise = denoise_method),
    by = c("Var2" = "fullID")
  )

# Join longData with metadf_method again to get Var2_denoise
longData <- longData %>%
  left_join(
    metadf_method %>% 
      dplyr::select(fullID, Var1_dataset = dataset),
    by = c("Var1" = "fullID")
  )

# Join longData with metadf_method again to get Var2_denoise
longData <- longData %>%
  left_join(
    metadf_method %>% 
      dplyr::select(fullID, Var2_dataset = dataset),
    by = c("Var2" = "fullID")
  )

# Join longData with metadf_method again to get Var2_denoise
longData <- longData %>%
  left_join(
    metadf_method %>% 
      dplyr::select(fullID, Var1_taxonomy_method = taxonomy_method),
    by = c("Var1" = "fullID")
  )

# Join longData with metadf_method again to get Var2_denoise
longData <- longData %>%
  left_join(
    metadf_method %>% 
      dplyr::select(fullID, Var2_taxonomy_method = taxonomy_method),
    by = c("Var2" = "fullID")
  )

In [0]:
longData$Var1 <- as.factor(longData$Var1)
longData$Var2 <- as.factor(longData$Var2)
str(longData)

In [0]:
longData <- longData %>%
  mutate(
    comparison = pmap_chr(
      list(Var1_denoise, Var2_denoise),
      function(v1, v2) {

        if (is.na(v1) | is.na(v2)) {
          NA_character_

        } else if (v1 == v2) {
          "Same denoise method"

        } else {
          paste(sort(c(v1, v2)), collapse = "")
        }
      }
    ),
    sameDenoise = Var1_denoise == Var2_denoise,
    sameData = Var1_dataset == Var2_dataset
  )

In [0]:
longData